# Qwen3-8B — Unsloth QLoRA
Mở notebook bằng VS Code → Select Kernel → Colab → GPU (T4 hoặc GPU CUDA khác).

1. Chạy cell cài thư viện. Nếu có cài mới, **Restart Kernel một lần**, rồi chạy lại cell đó để xác nhận.
2. Upload **KLTN_8B_Bundle.zip bản mới** vào thư mục gốc My Drive.
3. Chạy các cell tiếp theo từ trên xuống. Các siêu tham số nằm ở cell 2.
4. Train liên tục toàn bộ 1 epoch, không chia phiên. Lưu checkpoint sau bước 1, mỗi 25 optimizer steps và khi hoàn tất; sao lưu vào Drive sau mỗi lần lưu.
5. Muốn tiếp tục sau khi runtime bị mất: giữ nguyên cấu hình và RUN_NAME, đặt START_MODE="resume", chạy lại notebook. Có thể nhập RESUME_ZIP nếu đã sao chép một checkpoint sang Drive khác được phép sử dụng.

Dữ liệu cố định: **7.157 train / 583 validation / 465 test**; không thêm bản dịch OPUS-MT bị loại. Chỉ tính loss trên phần assistant; không cắt bỏ âm thầm mẫu dài. Nguồn chưa được chuyên gia chấm, dùng cho nghiên cứu/giáo dục. Loss thấp hoặc JSON hợp lệ chưa chứng minh câu viết hay hoặc đúng sự thật. Notebook không thay model đang phục vụ app.


In [8]:
# Kiểm tra kết nối Colab GPU
import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))

Tesla T4, 15360 MiB



In [9]:
# 1. Cài môi trường CUDA + Unsloth (chưa nạp model, chưa train)
import os, sys, subprocess, tempfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
CUDA_PACKAGES = ['torch==2.8.0+cu126', 'torchvision==0.23.0+cu126',
                 'torchaudio==2.8.0+cu126', 'xformers==0.0.32.post2']
PACKAGES = ['unsloth==2026.11.5', 'unsloth_zoo==2026.11.5', 'transformers==4.56.2', 'peft==0.18.1', 'trl==0.23.1', 'accelerate==1.10.1', 'bitsandbytes==0.50.2', 'datasets==4.3.0', 'torchao==0.13.0', 'numpy==2.2.6', 'huggingface_hub==0.36.2', 'sentencepiece==0.2.1']
def matches(pin):
    name, expected = pin.split('==')
    try: return version(name) == expected
    except PackageNotFoundError: return False
if not all(matches(pin) for pin in CUDA_PACKAGES + PACKAGES):
    if any(name in sys.modules for name in ('torch', 'transformers', 'peft', 'unsloth')):
        raise RuntimeError('Restart Kernel rồi chỉ chạy lại cell 1 trước.')
    def install(arguments):
        result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *arguments],
                                capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(result.stderr[-6000:])
    with tempfile.TemporaryDirectory() as temporary:
        constraint = Path(temporary) / 'constraints.txt'
        constraint.write_text('\n'.join(CUDA_PACKAGES + PACKAGES))
        install(['--index-url', 'https://download.pytorch.org/whl/cu126', *CUDA_PACKAGES])
        install(['-c', str(constraint), *PACKAGES])
    raise RuntimeError('Đã cài xong. Chọn Restart Kernel, chạy lại cell 1, rồi tiếp tục cell 2.')
print('Môi trường đúng phiên bản; tiếp tục cell 2.')


Môi trường đúng phiên bản; tiếp tục cell 2.


In [10]:
# 2. Đường dẫn, chế độ chạy và TOÀN BỘ siêu tham số
from pathlib import Path
ARCHIVE = Path('/content/drive/MyDrive/KLTN_8B_Bundle.zip')
WORK = Path('/content/kltn-8b-unsloth')
RUN_NAME = 'qwen3-8b-unsloth-v1'  # Đổi tên nếu bắt đầu một thí nghiệm khác.
PERSISTENT = Path('/content/drive/MyDrive/KLTN_8B') / RUN_NAME
START_MODE = 'resume'               # new: lượt mới; resume: khôi phục checkpoint.
RESUME_ZIP = '/content/drive/MyDrive/checkpoint-slot-0.zip'                # Hoặc '/content/drive/MyDrive/checkpoint-slot-0.zip'.

HYPERPARAMETERS = dict(
    max_length=2048,             # Tổng token prompt + đáp án.
    epochs=1.0,                  # Đi qua toàn bộ tập train một lần.
    learning_rate=5e-5,
    batch_size=1,                # Số mẫu mỗi forward/backward trên 1 GPU.
    gradient_accumulation_steps=8, # Effective batch = 1 × 8 × 1 GPU = 8.
    lora_rank=16,
    lora_alpha=32,               # Hệ số alpha/r = 2.
    lora_dropout=0.0,            # 0 để dùng đường tối ưu của Unsloth.
    target_modules=('q_proj', 'k_proj', 'v_proj', 'o_proj'),
    precision='float16',         # T4 dùng FP16; không tự đổi precision khi resume.
    optimizer='paged_adamw_8bit',
    warmup_ratio=0.03,
    scheduler='cosine',
    weight_decay=0.01,
    max_grad_norm=1.0,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    logging_steps=10,
    save_steps=25,               # Optimizer steps; thêm checkpoint ở bước 1 và cuối.
    save_total_limit=2,          # Giữ 2 checkpoint local; Drive giữ 3 bản luân phiên.
    eval_steps=100,              # Đánh giá trên 583 validation samples.
    seed=42,
    generation_max_new_tokens=384, # Chỉ cho phép thử sinh văn bản trước/sau train.
)


In [11]:
# 3. Mount Drive và giải nén dữ liệu/mã nguồn đã xác minh
import os, sys, json, hashlib, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')
# Nếu VS Code không hiện đăng nhập Drive: dùng Command Palette →
# "Colab: Mount Google Drive to Server...", hoàn tất xác thực rồi chạy lại cell này.
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload KLTN_8B_Bundle.zip bản mới vào My Drive: {ARCHIVE}')
if hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() != '5b9e862ce09911627dad8eb01404833cfe7f4086309b05494624357e6a2f09ad':
    raise ValueError('ZIP khác phiên bản notebook; upload lại ZIP mới trong ai/colab.')
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise ValueError('Invalid ZIP paths')
    archive.extractall(WORK)
for name, expected in json.loads((WORK / 'bundle-checksums.json').read_text()).items():
    if hashlib.sha256((WORK / name).read_bytes()).hexdigest() != expected:
        raise ValueError('File checksum mismatch: ' + name)
os.chdir(WORK)
if str(WORK) not in sys.path: sys.path.insert(0, str(WORK))
print('Bundle sẵn sàng. Backup:', PERSISTENT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bundle sẵn sàng. Backup: /content/drive/MyDrive/KLTN_8B/qwen3-8b-unsloth-v1


In [12]:
# 4. Nạp Qwen3-8B 4-bit + LoRA bằng Unsloth; kiểm tra dữ liệu và resume
from unsloth import FastLanguageModel  # Phải trước torch / transformers / peft.
from ai.colab.train_8b import (TrainConfig, load, datasets, run_identity, trainer,
                             evaluation_rows, probe, export_results, RUN)
from ai.colab.checkpoint_8b import require_drive, restore_checkpoint, atomic_json
CONFIG = TrainConfig(**HYPERPARAMETERS)
require_drive(PERSISTENT)
IDENTITY = run_identity(WORK, CONFIG)
CHECKPOINT = restore_checkpoint(WORK, PERSISTENT, IDENTITY, START_MODE, RESUME_ZIP)
TOKENIZER, MODEL, DTYPE = load(WORK, CONFIG)
DATA, DATA_MANIFEST = datasets(WORK, TOKENIZER, CONFIG)
TRAINER = trainer(WORK, TOKENIZER, MODEL, DATA, CONFIG, PERSISTENT, IDENTITY)
import math
steps = math.ceil(math.ceil(len(DATA['train']) / CONFIG.batch_size) /
                  CONFIG.gradient_accumulation_steps) * CONFIG.epochs
print('Effective batch:', CONFIG.batch_size * CONFIG.gradient_accumulation_steps,
      '| Dự kiến optimizer steps:', math.ceil(steps), '| Resume:', CHECKPOINT)


Restored optimizer step 525 from verified backup.
==((====))==  Unsloth 2026.11.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

trainable params: 15,335,424 || all params: 8,206,070,784 || trainable%: 0.1869
Tokenized samples: {'train': 7157, 'validation': 583, 'test': 465}
Effective batch: 8 | Dự kiến optimizer steps: 895 | Resume: /content/kltn-8b-unsloth/run-8b-unsloth/checkpoints/checkpoint-525


In [13]:
# 5. Giữ cùng 24 trường hợp trước/sau train; resume dùng lại baseline gốc
EVAL_ROWS = evaluation_rows(WORK)
if CHECKPOINT:
    BASELINE = json.loads((WORK / 'base-generation.json').read_text())
    original = json.loads((WORK / 'generation-eval-inputs.json').read_text())
    if original != EVAL_ROWS or [r['id'] for r in BASELINE] != [r['id'] for r in EVAL_ROWS]:
        raise ValueError('Baseline hoặc bộ đánh giá không khớp checkpoint.')
else:
    atomic_json(WORK / 'generation-eval-inputs.json', EVAL_ROWS)
    BASELINE = probe(WORK, TOKENIZER, MODEL, EVAL_ROWS, 'base-generation.json', CONFIG)
print('Baseline:', len(BASELINE), 'cases')


Baseline: 24 cases


In [14]:
# 6. TRAIN toàn bộ; mỗi checkpoint được sao lưu sang Drive trước khi đi tiếp
assert START_MODE == 'resume' and CHECKPOINT, 'Chưa khôi phục checkpoint; chạy lại cell 2 và 4.'
assert json.loads((Path(CHECKPOINT) / 'trainer_state.json').read_text())['global_step'] >= 525, 'Checkpoint cũ hơn bước 525; kiểm tra ZIP trên Drive.'
RESULT = TRAINER.train(resume_from_checkpoint=CHECKPOINT)
TRAINER.save_model(str(WORK / RUN / 'adapter'))
TOKENIZER.save_pretrained(WORK / RUN / 'adapter')
atomic_json(WORK / RUN / 'train-result.json', RESULT.metrics)
print('Train hoàn tất. Chạy cell 7 để đánh giá và lưu kết quả vào Drive.')


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,157 | Num Epochs = 1 | Total steps = 895
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 15,335,424 of 8,206,070,784 (0.19% trained)


Step,Training Loss,Validation Loss
600,0.611800,0.208636
700,0.500700,0.208218
800,0.507500,0.207110


Checkpoint 550 backed up: checkpoint-slot-1.zip
Checkpoint 575 backed up: checkpoint-slot-2.zip


Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Checkpoint 600 backed up: checkpoint-slot-0.zip
Checkpoint 625 backed up: checkpoint-slot-1.zip
Checkpoint 650 backed up: checkpoint-slot-2.zip
Checkpoint 675 backed up: checkpoint-slot-0.zip
Checkpoint 700 backed up: checkpoint-slot-1.zip
Checkpoint 725 backed up: checkpoint-slot-2.zip
Checkpoint 750 backed up: checkpoint-slot-0.zip
Checkpoint 775 backed up: checkpoint-slot-1.zip
Checkpoint 800 backed up: checkpoint-slot-2.zip
Checkpoint 825 backed up: checkpoint-slot-0.zip
Checkpoint 850 backed up: checkpoint-slot-1.zip
Checkpoint 875 backed up: checkpoint-slot-2.zip
Train hoàn tất. Chạy cell 7 để đánh giá và lưu kết quả vào Drive.


In [17]:
# 7. Đánh giá và xuất adapter + dữ liệu + cấu hình vào Drive
METRICS = TRAINER.evaluate(DATA['test'])
AFTER = probe(WORK, TOKENIZER, MODEL, EVAL_ROWS, 'adapter-generation.json', CONFIG)
atomic_json(WORK / RUN / 'evaluation.json', METRICS)
RESULT_ZIP = export_results(WORK, PERSISTENT)
print('Test loss:', METRICS.get('eval_loss'))
print('JSON hợp lệ:', sum(r['json_valid'] for r in BASELINE), '→',
      sum(r['json_valid'] for r in AFTER), '/ 24')
print('Kết quả đã lưu:', RESULT_ZIP)
# Xem các cặp trước/sau trong cell kế tiếp để đánh giá câu từ và tính đúng sự thật.


Test loss: 0.16745497286319733
JSON hợp lệ: 23 → 24 / 24
Kết quả đã lưu: /content/drive/MyDrive/KLTN_8B/qwen3-8b-unsloth-v1/KLTN_8B_Results.zip


In [16]:
# 8. Xem thử đầu ra; không tạo báo cáo riêng
for index in (12, 18):  # Một trường hợp writer và một polish tiếng Việt.
    source = json.loads(EVAL_ROWS[index]['messages'][1]['content'])
    print('\nDỮ LIỆU GỐC:', json.dumps(source, ensure_ascii=False))
    print('TRƯỚC:', BASELINE[index]['output'])
    print('SAU:', AFTER[index]['output'])



DỮ LIỆU GỐC: {"style": "professional", "max_bullets": 3, "section": "project", "instructions": "Câu rõ ràng, động từ hành động, mỗi bullet một ý.", "facts": [{"id": "f0", "kind": "work", "text": "mình đếm 17 mẫu cây và ghi vị trí của từng mẫu; không đánh giá chất lượng"}], "required_source_phrases": [{"source_id": "f0", "phrase": "không đánh giá chất lượng"}], "required_content_anchors": []}
TRƯỚC: {"bullets":[{"text":"Đếm 17 mẫu cây và ghi vị trí từng mẫu; không đánh giá chất lượng","source_ids":["f0"]}]}}
SAU: {"bullets":[{"text":"Đếm 17 mẫu cây và ghi vị trí của từng mẫu; không đánh giá chất lượng.","source_ids":["f0"]}]}

DỮ LIỆU GỐC: {"field": "description", "source_text": "mình đếm 50 mẫu cây và ghi vị trí của từng mẫu; không đánh giá chất lượng"}
TRƯỚC: {"text":"Đếm 50 mẫu cây và ghi vị trí của từng mẫu; không đánh giá chất lượng"}
SAU: {"text":"Đếm 50 mẫu cây và ghi vị trí của từng mẫu; không đánh giá chất lượng."}
